# 01 — Analyze LOL-v2 Low-Light Dataset

This notebook analyzes the **low-light images of LOL-v2-Synthetic** and builds a statistical reference profile for later VisDrone low-light synthesis.

## Outputs

The notebook produces:

- `LOL-v2_Profile.json` — aggregate luminance statistics and parameter reference
- `LOL-v2_Image_Statistics.csv` — per-image statistics
- `LOL-v2_Histograms.npy` — normalized luminance histograms
- `LOL-v2_PerImage_Profiles.npy` — per-image percentile/statistics profiles
- distribution plots and histogram visualizations

## Method

For every low-light image:

1. Read the image without resizing.
2. Convert BGR/RGB information to luminance using YCrCb.
3. Compute:
   - mean luminance
   - standard deviation
   - median
   - min/max
   - luminance percentiles
   - 256-bin histogram
   - entropy
   - dark-pixel ratios
4. Save per-image statistics.
5. Aggregate the dataset into a reusable reference profile.

> Important: images are analyzed at their original resolution to avoid losing small-image structures or altering the luminance distribution.


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

print("Imports loaded successfully.")


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

CONFIG = {
    # Path containing LOL-v2-Synthetic LOW-LIGHT images.
    # Change this path before running.
    "input_dir": "/path/to/LOL-v2-Synthetic/Low",

    # Output directory for all generated files.
    "output_dir": "./LOL-v2_Analysis_Output",

    # Supported image extensions.
    "extensions": [
        ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
    ],

    # Histogram configuration.
    "histogram_bins": 256,

    # Percentiles calculated for each image.
    "percentiles": [1, 5, 10, 25, 50, 75, 90, 95, 99],

    # Thresholds used to characterize darkness.
    # Luminance values are normalized to [0, 1].
    "dark_thresholds": [0.01, 0.03, 0.05, 0.10, 0.20],

    # Set to None to analyze all images.
    "max_images": None,

    # Save figures to the output directory.
    "save_figures": True,

    # Random seed used only for reproducible visualization sampling.
    "random_seed": 42,
}

np.random.seed(CONFIG["random_seed"])

INPUT_DIR = Path(CONFIG["input_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input directory :", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def list_images(directory, extensions):
    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(
            f"Input directory does not exist: {directory}"
        )

    extensions = {ext.lower() for ext in extensions}

    paths = [
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    ]

    return sorted(paths)


def load_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)

    if image is None:
        raise ValueError(f"Could not read image: {path}")

    return image


def extract_luminance_bgr(image_bgr):
    """
    Extract Y channel from OpenCV's YCrCb conversion.

    Returns:
        y_uint8: uint8 luminance in [0, 255]
        y: float32 luminance normalized to [0, 1]
    """

    ycrcb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2YCrCb
    )

    y_uint8 = ycrcb[:, :, 0]
    y = y_uint8.astype(np.float32) / 255.0

    return y_uint8, y


def shannon_entropy(hist):
    hist = np.asarray(hist, dtype=np.float64)
    hist = hist / (hist.sum() + 1e-12)

    hist = hist[hist > 0]

    return float(
        -np.sum(hist * np.log2(hist))
    )


def compute_image_statistics(
    image_path,
    histogram_bins=256,
    percentiles=(1, 5, 10, 25, 50, 75, 90, 95, 99),
    dark_thresholds=(0.01, 0.03, 0.05, 0.10, 0.20),
):
    image = load_image(image_path)

    height, width = image.shape[:2]

    y_uint8, y = extract_luminance_bgr(image)

    hist, _ = np.histogram(
        y,
        bins=histogram_bins,
        range=(0.0, 1.0),
        density=False,
    )

    hist_probability = (
        hist.astype(np.float64)
        / (hist.sum() + 1e-12)
    )

    record = {
        "image": str(image_path),
        "filename": image_path.name,
        "width": int(width),
        "height": int(height),
        "pixels": int(width * height),

        "mean_luminance": float(np.mean(y)),
        "std_luminance": float(np.std(y)),
        "variance_luminance": float(np.var(y)),
        "median_luminance": float(np.median(y)),
        "min_luminance": float(np.min(y)),
        "max_luminance": float(np.max(y)),

        "entropy": shannon_entropy(hist_probability),
    }

    percentile_values = np.percentile(
        y,
        percentiles
    )

    for p, value in zip(percentiles, percentile_values):
        record[f"p{int(p):02d}"] = float(value)

    for threshold in dark_thresholds:
        key = f"dark_ratio_le_{threshold:.2f}".replace(".", "_")

        record[key] = float(
            np.mean(y <= threshold)
        )

    return record, hist_probability


print("Helper functions ready.")


In [ ]:
# ============================================================
# 4. DISCOVER LOL-v2 LOW-LIGHT IMAGES
# ============================================================

image_paths = list_images(
    INPUT_DIR,
    CONFIG["extensions"]
)

if CONFIG["max_images"] is not None:
    image_paths = image_paths[:CONFIG["max_images"]]

print(f"Found {len(image_paths)} images.")

if len(image_paths) == 0:
    print(
        "\nWARNING: No images found. "
        "Please check CONFIG['input_dir']."
    )
else:
    print("\nFirst 5 images:")
    for path in image_paths[:5]:
        print(" -", path)


In [ ]:
# ============================================================
# 5. ANALYZE EVERY IMAGE
# ============================================================

records = []
histograms = []
failed_images = []

for image_path in tqdm(
    image_paths,
    desc="Analyzing LOL-v2"
):
    try:
        record, histogram = compute_image_statistics(
            image_path=image_path,
            histogram_bins=CONFIG["histogram_bins"],
            percentiles=CONFIG["percentiles"],
            dark_thresholds=CONFIG["dark_thresholds"],
        )

        records.append(record)
        histograms.append(histogram)

    except Exception as error:
        failed_images.append({
            "image": str(image_path),
            "error": str(error),
        })

print(f"Successfully analyzed: {len(records)}")
print(f"Failed images:        {len(failed_images)}")


In [ ]:
# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(records)

if len(df) == 0:
    raise RuntimeError(
        "No valid images were analyzed. "
        "Check the input path and image files."
    )

display(
    df.head()
)

print("\nDataset summary:")
print(df.shape)


In [ ]:
# ============================================================
# 7. SAVE PER-IMAGE STATISTICS
# ============================================================

csv_path = OUTPUT_DIR / "LOL-v2_Image_Statistics.csv"

df.to_csv(
    csv_path,
    index=False
)

print("Saved:")
print(csv_path)


In [ ]:
# ============================================================
# 8. SAVE HISTOGRAM ARRAYS
# ============================================================

histograms_array = np.asarray(
    histograms,
    dtype=np.float32
)

histogram_path = OUTPUT_DIR / "LOL-v2_Histograms.npy"

np.save(
    histogram_path,
    histograms_array
)

print("Histogram array shape:", histograms_array.shape)
print("Saved:")
print(histogram_path)


In [ ]:
# ============================================================
# 9. CREATE PER-IMAGE NUMERICAL PROFILE ARRAY
# ============================================================

profile_columns = [
    "mean_luminance",
    "std_luminance",
    "median_luminance",
    "min_luminance",
    "max_luminance",
    "entropy",
] + [
    f"p{int(p):02d}"
    for p in CONFIG["percentiles"]
]

per_image_profiles = (
    df[profile_columns]
    .to_numpy(dtype=np.float32)
)

profile_array_path = (
    OUTPUT_DIR
    / "LOL-v2_PerImage_Profiles.npy"
)

np.save(
    profile_array_path,
    per_image_profiles
)

print("Per-image profile shape:", per_image_profiles.shape)
print("Columns:")
print(profile_columns)
print("\nSaved:")
print(profile_array_path)


In [ ]:
# ============================================================
# 10. AGGREGATE LOL-v2 DATASET PROFILE
# ============================================================

def summarize_column(series):
    return {
        "mean": float(series.mean()),
        "std": float(series.std(ddof=0)),
        "min": float(series.min()),
        "max": float(series.max()),
        "median": float(series.median()),
        "p05": float(series.quantile(0.05)),
        "p25": float(series.quantile(0.25)),
        "p75": float(series.quantile(0.75)),
        "p95": float(series.quantile(0.95)),
    }


summary_columns = [
    "mean_luminance",
    "std_luminance",
    "variance_luminance",
    "median_luminance",
    "min_luminance",
    "max_luminance",
    "entropy",
] + [
    f"p{int(p):02d}"
    for p in CONFIG["percentiles"]
]

dark_ratio_columns = [
    column
    for column in df.columns
    if column.startswith("dark_ratio_")
]


aggregate_statistics = {
    column: summarize_column(df[column])
    for column in summary_columns
}

dark_pixel_statistics = {
    column: summarize_column(df[column])
    for column in dark_ratio_columns
}


average_histogram = (
    histograms_array.mean(axis=0)
)

histogram_std = (
    histograms_array.std(axis=0)
)


dataset_profile = {
    "dataset": "LOL-v2-Synthetic Low-Light",
    "num_images": int(len(df)),

    "image_dimensions": {
        "width": summarize_column(df["width"]),
        "height": summarize_column(df["height"]),
        "pixels": summarize_column(df["pixels"]),
    },

    "statistics": aggregate_statistics,

    "dark_pixel_ratios": dark_pixel_statistics,

    "histogram": {
        "bins": int(CONFIG["histogram_bins"]),
        "range": [0.0, 1.0],
        "average": average_histogram.tolist(),
        "std": histogram_std.tolist(),
    },

    "per_image_profile_columns": profile_columns,

    "analysis_configuration": {
        "percentiles": CONFIG["percentiles"],
        "dark_thresholds": CONFIG["dark_thresholds"],
        "luminance_space": "OpenCV YCrCb Y channel normalized to [0, 1]",
        "images_resized": False,
    },
}

json_path = OUTPUT_DIR / "LOL-v2_Profile.json"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(
        dataset_profile,
        file,
        indent=4
    )

print("Saved aggregate profile:")
print(json_path)


In [ ]:
# ============================================================
# 11. DISPLAY KEY LOL-v2 REFERENCE VALUES
# ============================================================

summary_table = pd.DataFrame(
    {
        metric: values
        for metric, values
        in aggregate_statistics.items()
    }
).T

display(
    summary_table.round(6)
)


In [ ]:
# ============================================================
# 12. VISUALIZE MEAN LUMINANCE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    df["mean_luminance"],
    bins=40
)

plt.xlabel("Mean luminance")
plt.ylabel("Number of images")
plt.title("LOL-v2-Synthetic: Distribution of Mean Luminance")
plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "Mean_Luminance_Distribution.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 13. VISUALIZE LUMINANCE STANDARD DEVIATION
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    df["std_luminance"],
    bins=40
)

plt.xlabel("Luminance standard deviation")
plt.ylabel("Number of images")
plt.title("LOL-v2-Synthetic: Distribution of Luminance Contrast")
plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "Luminance_STD_Distribution.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 14. AVERAGE LUMINANCE HISTOGRAM
# ============================================================

bin_centers = np.linspace(
    0,
    1,
    CONFIG["histogram_bins"],
    endpoint=False
)

plt.figure(figsize=(9, 5))

plt.plot(
    bin_centers,
    average_histogram,
    label="Average histogram"
)

plt.fill_between(
    bin_centers,
    np.maximum(
        0,
        average_histogram - histogram_std
    ),
    average_histogram + histogram_std,
    alpha=0.2,
    label="± 1 std"
)

plt.xlabel("Normalized luminance")
plt.ylabel("Probability")
plt.title("LOL-v2-Synthetic: Average Low-Light Luminance Distribution")
plt.legend()
plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "Average_Luminance_Histogram.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 15. PERCENTILE PROFILE
# ============================================================

percentile_columns = [
    f"p{int(p):02d}"
    for p in CONFIG["percentiles"]
]

percentile_means = [
    df[column].mean()
    for column in percentile_columns
]

percentile_stds = [
    df[column].std(ddof=0)
    for column in percentile_columns
]

plt.figure(figsize=(8, 5))

plt.plot(
    CONFIG["percentiles"],
    percentile_means,
    marker="o",
    label="Mean percentile value"
)

plt.fill_between(
    CONFIG["percentiles"],
    np.maximum(
        0,
        np.array(percentile_means)
        - np.array(percentile_stds)
    ),
    np.array(percentile_means)
    + np.array(percentile_stds),
    alpha=0.2,
    label="± 1 std"
)

plt.xlabel("Image luminance percentile")
plt.ylabel("Normalized luminance")
plt.title("LOL-v2-Synthetic: Dataset Percentile Profile")
plt.legend()
plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "Percentile_Profile.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 16. DARK PIXEL RATIO ANALYSIS
# ============================================================

dark_summary = pd.DataFrame({
    "metric": dark_ratio_columns,
    "mean_ratio": [
        df[column].mean()
        for column in dark_ratio_columns
    ],
    "std_ratio": [
        df[column].std(ddof=0)
        for column in dark_ratio_columns
    ],
})

display(
    dark_summary.round(6)
)

plt.figure(figsize=(9, 5))

plt.bar(
    dark_summary["metric"],
    dark_summary["mean_ratio"]
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean pixel ratio")
plt.title("LOL-v2-Synthetic: Dark Pixel Ratios")
plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "Dark_Pixel_Ratios.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 17. VISUALIZE RANDOM LOW-LIGHT EXAMPLES
# ============================================================

sample_count = min(8, len(image_paths))

sample_indices = np.linspace(
    0,
    len(image_paths) - 1,
    sample_count,
    dtype=int
)

plt.figure(figsize=(16, 8))

for index, image_index in enumerate(sample_indices, start=1):

    image = load_image(
        image_paths[image_index]
    )

    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    plt.subplot(2, 4, index)
    plt.imshow(image_rgb)
    plt.title(
        image_paths[image_index].name,
        fontsize=8
    )
    plt.axis("off")

plt.suptitle(
    "LOL-v2-Synthetic Low-Light Samples",
    fontsize=14
)

plt.tight_layout()

if CONFIG["save_figures"]:
    path = OUTPUT_DIR / "LowLight_Samples.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

plt.show()


In [ ]:
# ============================================================
# 18. OPTIONAL: SAVE FAILED IMAGE LOG
# ============================================================

if failed_images:

    failed_path = (
        OUTPUT_DIR
        / "LOL-v2_Failed_Images.json"
    )

    with open(
        failed_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            failed_images,
            file,
            indent=4
        )

    print(
        "Saved failed image log:",
        failed_path
    )

else:

    print(
        "No failed images."
    )


# 19. Output Interpretation

The most important outputs for the next notebook (`02_optimize_visdrone_lowlight.ipynb`) are:

## `LOL-v2_Profile.json`

Use this as the dataset-level reference for:

- target mean luminance
- target luminance standard deviation
- percentile ranges
- dark-pixel ratios
- average luminance histogram

## `LOL-v2_Image_Statistics.csv`

Use this when sampling a realistic **individual target profile** rather than forcing every VisDrone image toward one global average.

## `LOL-v2_Histograms.npy`

Use this for histogram-distance optimization, for example:

- Wasserstein distance
- Jensen-Shannon divergence
- Earth Mover's Distance

## `LOL-v2_PerImage_Profiles.npy`

Use this for sampling image-level targets during VisDrone synthesis.

---

## Recommended strategy for Notebook 02

For each clear VisDrone image:

1. Select or sample a LOL-v2 target profile.
2. Generate candidate low-light parameters.
3. Apply luminance-only degradation.
4. Preserve edges and small-object structures.
5. Measure the distance between the synthetic VisDrone profile and the selected LOL-v2 profile.
6. Keep the best parameter set.

This produces a statistically guided low-light synthesis pipeline rather than relying on fixed heuristic gamma values.
